# Week 4: The Propensity Score, Weighting and Doubly Robust Estimation

This notebook follows the Week 4 page, in the same order:
<https://yoonismo.github.io/causal-computing/week4.html>

Read the explanations on the page and run the cells here **in order** from top to bottom. In Lab 6 you paste code from your AI assistant into a cell marked `# TODO`, followed by a **check** cell. Write your answers to the questions in the empty text cells.

## 0. Before you start

- Check **Select Kernel** at the top right: it should say **ECON 481 (Python 3.12)**.
- If a cell fails with `ModuleNotFoundError`, a package is missing from your environment. See `README.md`.
- The data sets are downloaded from the internet when you load them, so you need a connection.
- A check cell prints `not done yet` until you fill in the `# TODO` cell above it. Every cell must still run from top to bottom when you push: GitHub runs this notebook after every push.

First, we fix the random seed for reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(481)

## From Week 3 to Week 4

In Week 3 we identified causal effects without a randomized experiment under the three identification assumptions, consistency, unconfoundedness, and positivity:

- **consistency:** $Y_i = D_i Y_i(1) + (1-D_i)Y_i (0)$
- **unconfoundedness:** $(Y_i(1), Y_i(0)) \perp D_i \mid X_i$, and
- **common support:** $0 < P(D_i = 1 \mid X_i = x) < 1$ for every $x$. (a.k.a. positivity)

They are critical assumptions for identification. Causal inference literature has developed numerous methods to measure causal effects. The main approach is to abide by the above identification assumptions by literally following what they mean: match units whose covariates of a group are equal or similar to those of one another. Subclassification and matching are the estimation methods that satisfy it. However, the two approaches have problems when it comes to many covariates. 

1. Subclassification encounters a challenge because cells in groups become empty as the number of covariates grows. So, it requires more observations to not have empty cells, a.k.a. the curse of dimensionality.
2. Matching methods have difficulty finding matches in a group whose covariates are similar to those of their counterparts as the dimension of covariates increases since it becomes hard to find the matches **close enough** to the ones in every or even most aspects in covariates.

Hence, this week we are going to learn two estimators to resolve the above issues and others:

1. **inverse probability weighting** estimators, through the **propensity score**, to estimate causal parameters such as the ATE and/or ATT.  
2. **augmented inverse probability weighting** estimators to handle model misspecification errors, combined with the IPW. 

## The propensity score

So, what is the propensity score? First, let me introduce its definition. 

**Definition** (Rosenbaum and Rubin, 1983) The propensity score is the conditional probability of assignment to the treatment group given an observed covariate $X_i = x$, 
$$
e(x) = P(D_i = 1 \mid X_i = x).
$$

What does this mean? It is a measure of the share of units in the treatment group given a vector of observed covariates out of all the units with the same covariates (Cunningham, 2026). We can view it as a number between 0 and 1 that consolidates all the information from observed covariates. This means that we don't need to compute the discrepancy between covariates in two groups one by one, but only need to compute one value that contains all the information about treatment assignments relevant to measure causal effects. 

Now it is natural that we have the following questions: 

1. How do we compute or **estimate** the propensity score? 
2. Where and how do we use it in our potential outcomes framework? 



## Lab 1 — Estimating the propensity score

Look at the propensity score, $e(x)$, again. What we know is that it is a probability, but we don't know how to compute it. Thus, we need a specific functional form to estimate it. 

First, we call for data.

In [ ]:
from scipy.optimize import minimize
from causaldata import nsw_mixtape, cps_mixtape

nsw = nsw_mixtape.load_pandas().data
cps = cps_mixtape.load_pandas().data

treated = nsw[nsw["treat"] == 1]
control = nsw[nsw["treat"] == 0]
benchmark = treated["re78"].mean() - control["re78"].mean()    # experimental estimate (Week 2)

# one data set with the trainees and the CPS men
both = pd.concat([treated.assign(d=1), cps.assign(d=0)], ignore_index=True)
both["u74"] = (both["re74"] == 0).astype(int)    # 1 if no earnings in 1974
both["u75"] = (both["re75"] == 0).astype(int)    # 1 if no earnings in 1975
both["re74k"] = both["re74"] / 1000              # earnings in thousands of dollars
both["re75k"] = both["re75"] / 1000

simple_vars = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74k", "re75k"]
X = np.column_stack([np.ones(len(both)), both[simple_vars].to_numpy(dtype=float)])   # n x (1 + 8), first column = 1
D = both["d"].to_numpy(dtype=float)

### 1.1 Examples of functional forms to estimate the propensity score

In practice, econometricians and many scientists use specific functional forms such as logit or probit.

#### 1.1.1 Logit model

We model the propensity score with a **logit**:

$$
e_{\gamma}(x) = \text{Pr} (D_i =1 | X_i = x; \gamma)= \frac{1}{1 + \exp(- (\gamma_0  + x'\gamma_{*}))},
$$

where $x$ is a $d$-dimensional vector in the Euclidean space, $\mathbb{R}^d$. Notice that if we know the coefficients $\gamma = [\gamma_0, \gamma_{*}]^{\top} = [\gamma_0, \gamma_1, \gamma_2, ...\gamma_d]^{\top}$, we can compute the propensity score. How do we compute it? We construct the likelihood function $\mathcal{L}(\gamma; X):$ 
$$
    \mathcal{L}(\gamma; X) = \prod_{i=1}^n [e(X_i)]^{D_i} [1-e(X_i)]^{1-D_i} =  \prod_{i=1}^n [(1 + \exp(- X_i^{\top}\gamma))^{-1}]^{D_i} [1-(1 + \exp(- X_i^{\top}\gamma))^{-1}]^{1-D_i}
$$
We take the log on $\mathcal{L}(\gamma; X)$ to make our lives easier. We maximize the log-likelihood $\ell(\gamma ; X) = \log \mathcal{L}(\gamma; X)$:

$$
\ell(\gamma) = \sum_{i=1}^{n} \Big[ D_i \log e(X_i) + (1 - D_i) \log\big(1 - e(X_i)\big) \Big].
$$

Even though we seem to have a nice objective function to solve for $\gamma$, there is no formula for the maximizer. Thus we find it numerically: `scipy.optimize.minimize` minimizes $-\ell(\gamma)$. We start with a simple model that has each covariate once.

In [ ]:
def neg_loglik(gamma):
    e = 1 / (1 + np.exp(-X @ gamma))                # propensity score of every person
    e = np.clip(e, 1e-12, 1 - 1e-12)                # avoid log(0)
    return -np.sum(D * np.log(e) + (1 - D) * np.log(1 - e))

fit = minimize(neg_loglik, x0=np.zeros(X.shape[1]), method="BFGS")
gamma_hand = fit.x
pd.Series(gamma_hand, index=["const"] + simple_vars).round(3)

#### 1.1.2 Probit model

We can also use the probit to model the propensity score:

$$
e(x) = \text{Pr} (D_i =1 | X_i = x ; \gamma)= \Phi (x^{\top}\gamma) = \int_{-\infty}^{x^{\top}\gamma} \frac{1}{\sqrt{2 \pi}}\exp(-\frac{1}{2} z^2)dz ,
$$

Following the same process as in **1.1.1**, we can construct the likelihood with respect to the probit: 
$$
    \mathcal{L}(\gamma; X) = \prod_{i=1}^n [e(X_i)]^{D_i} [1-e(X_i)]^{1-D_i} =  \prod_{i=1}^n [\Phi (X_i^{\top}\gamma)]^{D_i} [1-\Phi (X_i^{\top}\gamma)]^{1-D_i}
$$
We take the log on $\mathcal{L}(\gamma; X)$ again and maximize the log-likelihood $\ell(\gamma ; X) = \log \mathcal{L}(\gamma; X)$:

$$
\ell(\gamma) = \sum_{i=1}^{n} \Big[ D_i \log e(X_i) + (1 - D_i) \log\big(1 - e(X_i)\big) \Big].
$$

Once again there is no formula for the maximizer. Thus we find it numerically: `scipy.optimize.minimize` minimizes $-\ell(\gamma)$. We start with a simple model that has each covariate once.

In [ ]:
from scipy.stats import norm

def neg_loglik_probit(gamma):
    return -np.sum(D * norm.logcdf(X @ gamma) + (1 - D) * norm.logcdf(- X @ gamma))

fit = minimize(neg_loglik_probit, x0=np.zeros(X.shape[1]), method="BFGS")
gamma_hand_probit = fit.x
pd.Series(gamma_hand_probit, index=["const"] + simple_vars).round(3)

In [ ]:
import statsmodels.formula.api as smf

logit_simple = smf.logit("d ~ " + " + ".join(simple_vars), data=both).fit(disp=0)
probit_simple = smf.probit("d ~ " + " + ".join(simple_vars), data=both).fit(disp=0)


pd.DataFrame({
    "logit, by hand": gamma_hand,
    "logit, statsmodels": logit_simple.params.to_numpy(),
    "probit, by hand": gamma_hand_probit,
    "probit, statsmodels": probit_simple.params.to_numpy(),
}, index=["const"] + simple_vars).round(3)

### 1.2 A more flexible model

If we look at the argument of both models, we use a linear combination of each covariate with the coefficient $\gamma$. However, the covariates may have different forms such as polynomials of finite degree. Following the spirit of Dehejia and Wahba (1999), we add squares, a cube of age, the zero-earnings indicators and an interaction. We use this model for the rest of the week.

In [ ]:
ps_formula = ("d ~ age + I(age**2) + I(age**3) + educ + I(educ**2) + marr + nodegree + black + hisp"
              " + re74k + re75k + u74 + u75 + educ:re74k")
logit = smf.logit(ps_formula, data=both).fit(disp=0)
probit = smf.probit(ps_formula, data=both).fit(disp=0)
both["e_logit"] = logit.predict(both)  
both["e"] = both["e_logit"]
both["e_probit"] = probit.predict(both)  

t = both[both["d"] == 1]                         # trainees, used in the rest of the notebook
c = both[both["d"] == 0]                         # CPS men
covariates = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]

both.groupby("d")[["e_logit", "e_probit"]].describe().round(3)

### 1.3 Common support

The estimated scores of the trainees and the CPS men, on a log scale because most CPS men have scores very close to 0.

In [ ]:
bins = np.logspace(-7, 0, 36)                    # bins of equal width on the log scale
for group, color, label in [(1, "#2a78d6", "NSW trainees"), (0, "#eb6834", "CPS men")]:
    scores = both.loc[both["d"] == group, "e"]
    plt.hist(scores, bins=bins, weights=np.full(len(scores), 1 / len(scores)),   # each bar = share of the group
             histtype="step", linewidth=2, color=color, label=label)
plt.xscale("log")
plt.xlabel("Estimated propensity score (log scale)")
plt.ylabel("Share of the group")
plt.legend()
plt.show()

pd.crosstab(pd.cut(both["e"], [0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1]), both["d"]).rename(
    columns={0: "CPS men", 1: "trainees"})

**Question 1.** 
- Where are most CPS men, and where are most trainees? 
- Is there enough overlap to estimate the effect for people like the trainees (ATT)? 
- For everyone in the pooled sample (ATE)?

*Your answer to Question 1:*

## Inverse probability weighting (IPW)

In a randomized experiment everyone has the same chance of treatment regardless of who they are. However, in observational studies the probability of getting treated depends on their characteristics. That is, some people are more likely to be treated than others. 

How do we resolve this issue? We use the propensity score $e(X_i)$. It mitigates the issues from subclassification and matching methods. 

Then how exactly do we use the propensity score? By weighting each treated person by $1/e(X_i)$ and each untreated person by $1/(1 - e(X_i))$. 

What does this mean? The treated units whose propensity scores are high or close to 1 will receive less weight than other treated units with smaller propensity scores. If a unit's propensity score is close to 1, it can be viewed as "obvious," because given an observed covariate $X_i = x$, he or she has more chance to get treated. However, if there is a treated unit whose probability of being treated is low, the chance of being treated can be viewed as "rare," because the unit was treated although the unit's characteristics showed they would not be likely to be. 

IPW is an estimator of a causal effect adjusting for weighting for each outcome value. 

### ATE in terms of IPW

We derive the causal parameter ATE, $\tau_{\text{ATE}}$ using the IPW representation. Note that from the consistency assumption, 

$$
    Y_i = D_i Y_i(1) + (1-D_i) Y_i(0)
$$
we see that 

$$
    D_i Y_i = D_i^2  Y_i(1) + D_i (1-D_i) Y_i(0) = D_i Y_i(1) \tag{*}
$$
because $ D_i (1-D_i) Y_i(0) = 0$ regardless of $D_i$. In addition, $D_i^2 = D_i $ when $D_i$ takes values only from $\{0,1\}$. 

The counterpart also holds:
$$
   (1-D_i)Y_i = (1-D_i) D_i Y_i(1) + (1-D_i)^2 Y_i(0) = (1-D_i) Y_i(0) \tag{**}
$$

We are now ready to derive $\tau_{\text{ATE}}$ using the IPW representation.
$$
\begin{align*}
    \tau_{\text{ATE}} =& E[Y_i(1)-Y_i(0)] \\ 
                      =& E\big[E[Y_i(1)-Y_i(0) | X_i]\big] \tag{Law of iterated expectations}\\ 
                      =& E\big[E[Y_i(1)| X_i] - E[Y_i(0) | X_i]\big] \tag{***}\\
                      =& E\big[E[Y_i(1)| X_i]\frac{E[D_i | X_i]}{E[D_i | X_i]} - E[Y_i(0) | X_i] \frac{E[(1-D_i) | X_i]}{E[(1-D_i) | X_i]}\big] \\
                      =& E\big[ \frac{1}{E[D_i | X_i]}E[Y_i(1)| X_i]E[D_i | X_i] - \frac{1}{E[(1-D_i) | X_i]}E[Y_i(0) | X_i]E[(1-D_i) | X_i]\big] \\
                      =& E\big[ \frac{1}{e(X_i)}E[Y_i(1)| X_i]E[D_i | X_i] - \frac{1}{(1 -e(X_i))}E[Y_i(0) | X_i]E[(1-D_i) | X_i]\big] \\
                      =& E\big[ \frac{1}{e(X_i)}E[D_iY_i(1)| X_i] - \frac{1}{(1 -e(X_i))}E[(1-D_i) Y_i(0) | X_i]\big] \\
                      =& E\big[ \frac{1}{e(X_i)}E[D_iY_i| X_i] - \frac{1}{(1 -e(X_i))}E[(1-D_i) Y_i | X_i]\big] \tag{(*) and (**)}\\
                      =& E[ \frac{D_iY_i}{e(X_i)} - \frac{(1-D_i) Y_i }{(1 -e(X_i))}]
\end{align*}
$$

For ATT, 

$$
\begin{align*}
    \tau_{\text{ATT}} =& E[Y_i(1)-Y_i(0)| D_i = 1] \\ 
                      =& E[Y_i(1)|  D_i = 1] - E[Y_i(0)| D_i = 1] \\
                      =& E[ \frac{1}{\text{Pr}(D_i = 1)} E[ Y_i(1)|X_i ]  e(X_i)] - E[ \frac{1}{\text{Pr}(D_i = 1)} E[ Y_i(0)|X_i ] e(X_i)]\tag{Apply (***)}\\
                      =& \frac{1}{\text{Pr}(D_i = 1)}\big[E[  E[ Y_i(1)|X_i ]  e(X_i)] - E[  E[ Y_i(0)|X_i ]  e(X_i)]\big]\\
                      =& \frac{1}{\text{Pr}(D_i = 1)}\big[E[  \frac{D_i Y_i}{e(X_i)}  e(X_i)] - E[  \frac{(1-D_i) Y_i}{1-e(X_i)}   e(X_i)]\big]\\
                      =&\frac{1}{\text{Pr}(D_i = 1)}E[   e(X_i) \big\{\frac{D_iY_i}{e(X_i)} - \frac{(1-D_i) Y_i }{(1 -e(X_i))}\big\}]\\
\end{align*}
$$

**In the sample** we replace the expectations by sample averages: 

$$
    \hat{\tau}_{\text{ATE}}^{\text{IPW}} = \frac{1}{n}\sum_{i=1}^n \bigg(\frac{D_iY_i}{\hat{e}(X_i)} - \frac{(1-D_i) Y_i }{(1 -\hat{e}(X_i))}\bigg) \qquad \hat{\tau}_{\text{ATT}}^{\text{IPW}} = \frac{1}{n_1}\sum_{i=1}^n \hat{e}(X_i)\bigg(\frac{D_iY_i}{\hat{e}(X_i)} - \frac{(1-D_i) Y_i }{(1 -\hat{e}(X_i))}\bigg)
$$

However, these simple plug-in estimators, **Horvitz–Thompson** estimator from survey sampling (Horvitz and Thompson, 1952), are sensitive to extreme values of the propensity score. Hence, some researchers proposed an alternative to the sample analog estimators above. Hirano and Imbens (2001) used the following: 

$$
    \hat{\tau}_{\text{ATE}}^{\text{IPW}} =  \bigg[ \sum_{i=1}^n \frac{D_iY_i}{\hat{e}(X_i)}/ (\sum_{i=1}^n \frac{D_i}{\hat{e}(X_i)})\bigg] - \bigg[ \sum_{i=1}^n \frac{(1-D_i) Y_i }{(1 -\hat{e}(X_i))}/ (\sum_{i=1}^n \frac{(1-D_i)}{(1-\hat{e}(X_i))})\bigg] 
$$

$$
    \hat{\tau}_{\text{ATT}}^{\text{IPW}} = \big(\sum_{i =1}^n D_iY_i\big) / (\sum_{i=1}^n D_i) - \big(\sum_{i=1}^n \hat{w}_i (1-D_i) Y_i \big) / (\sum_{i=1}^n \hat{w}_i (1-D_i)) 
$$

where $\hat{w}_i = \hat{e}(X_i)/(1- \hat{e}(X_i)) $. It is also called the  **Hájek** estimator (Hájek, 1971).



## Lab 2 — Inverse probability weighting

### 2.1 A world where we know the truth

We build a world with one covariate $X_i \sim N(0, 1)$ (think of a test score), so that we can check the estimators against the truth. Treatment depends on $X_i$ through a logit with a squared term, the outcome without treatment is quadratic in $X_i$, and the effect grows with $X_i$:

$$
e(x) = \frac{1}{1 + \exp\{-(-0.3 + 0.8 x + 0.2 x^2)\}}, \qquad
Y_i(0) = 8 + 2 X_i + 1.5 X_i^2 + \varepsilon_i, \qquad
Y_i(1) = Y_i(0) + 2 + X_i ,
$$

with $\varepsilon_i \sim N(0, 2^2)$. People with a high $X$ are more often treated **and** gain more, so the ATT is larger than the ATE (which is 2).

In [ ]:
n = 10_000
x = rng.normal(0, 1, n)
e_true = 1 / (1 + np.exp(-(-0.3 + 0.8 * x + 0.2 * x**2)))   # true propensity score
d = rng.binomial(1, e_true)
y0 = 8 + 2 * x + 1.5 * x**2 + rng.normal(0, 2, n)
y1 = y0 + 2 + x                                             # effect 2 + x: larger for high x
y = d * y1 + (1 - d) * y0

print("True ATE:         ", round((y1 - y0).mean(), 3))
print("True ATT:         ", round((y1 - y0)[d == 1].mean(), 3))
print("Simple difference:", round(y[d == 1].mean() - y[d == 0].mean(), 3))

IPW with an **estimated** score, a logit of $D_i$ on $X_i$ and $X_i^2$, written out with the formulas above:

In [ ]:
import statsmodels.api as sm

Z = sm.add_constant(np.column_stack([x, x**2]))
e_hat = sm.Logit(d, Z).fit(disp=0).predict(Z)   # estimated propensity score

w1 = d / e_hat                          # weights of the treated for the ATE
w0 = (1 - d) / (1 - e_hat)              # weights of the untreated for the ATE
ate_ipw = (w1 * y).sum() / w1.sum() - (w0 * y).sum() / w0.sum()

odds = (1 - d) * e_hat / (1 - e_hat)    # weights of the untreated for the ATT
att_ipw = y[d == 1].mean() - (odds * y).sum() / odds.sum()

print("IPW, ATE:", round(ate_ipw, 3))
print("IPW, ATT:", round(att_ipw, 3))

**Question 2.** In the ATE derivation, which equality uses unconfoundedness, and where would the formula break if $e(x) = 0$ for some $x$? Why does the ATT need only $e(x) < 1$?

*Your answer to Question 2:*

### 2.2 IPW on the NSW–CPS data

With the estimated score from Lab 1. The CPS men get the odds weights $\hat e_i / (1 - \hat e_i)$.

In [ ]:
omega = c["e"] / (1 - c["e"])                     # odds weights of the CPS men
att_ipw_nsw = t["re78"].mean() - (omega * c["re78"]).sum() / omega.sum()

ate_ipw_nsw = ((t["re78"] / t["e"]).sum() / (1 / t["e"]).sum()
               - (c["re78"] / (1 - c["e"])).sum() / (1 / (1 - c["e"])).sum())

print("IPW, ATT:", round(att_ipw_nsw, 1), "  (benchmark", round(benchmark, 1), ")")
print("IPW, ATE:", round(ate_ipw_nsw, 1))
print("Largest weight 1/e among the trainees:", round((1 / t["e"]).max(), 1))

Balance after weighting: the trainees' means against the **weighted** means of the CPS men.

In [ ]:
pd.DataFrame({
    "trainees": t[covariates].mean(),
    "weighted CPS": c[covariates].mul(omega, axis=0).sum() / omega.sum(),
    "all CPS": c[covariates].mean(),
}).round(2)

**Question 3.** The ATT is in the right range, but the ATE is far off. Using the overlap table of Lab 1, explain why. Which trainees receive the largest weights in the ATE, and what does that do to the estimate?

*Your answer to Question 3:*

## Outcome model with a regression 

Let's say we are interested in estimating the ATE. We know the definition of ATE:
$$
    \tau_{\text{ATE}} = E[Y_i(1)-Y_i(0)].
$$

By the law of iterated expectations, we have 

$$
    \tau_{\text{ATE}} = E[Y_i(1)-Y_i(0)] = E[E[Y_i(1)-Y_i(0)| X_i]] = E[E[Y_i(1) | X_i]- E[Y_i(0)| X_i]]
$$

Take a look at the first term. If we do some algebra using the unconfoundedness assumption, we can derive the following: 

$$
    E[Y_i(1) \mid X_i] =  E[Y_i(1) \mid D_i = 1, X_i] = E[Y_i | D_i =1, X_i]
$$

It seems familiar to us. It is a regression model! We define a linear regression model to study the effect of regressors $X_i$ on the outcome variable $Y_i$ with an error term $\epsilon_i$ with mean zero conditional on $X_i$, such that

$$ 
\begin{align*}
  &Y_i = \beta_0 + \beta_1 X_1 + ... + \beta_d X_d + \epsilon_i\\
  E[&Y_i | X_i] = \beta_0 + \beta_1 X_1 + ... + \beta_d X_d  
\end{align*}
$$

If we add $D_i$, then we have $Y_i = \tau D_i + \beta_0 + \beta_1 X_1 + ... + \beta_d X_d + \epsilon_i$, and when $D_i = 1$ and $\tau$ is constant across units (homogenous treatment effect),

$$ 
\begin{align*}
  &Y_i = \tau_{\text{ATE}} D_i + \beta_0 + \beta_1 X_1 + ... + \beta_d X_d + \epsilon_i\\
  E[&Y_i | D_i = 1, X_i] = \tau_{\text{ATE}}+ \beta_0 + \beta_1 X_1 + ... + \beta_d X_d.  
\end{align*}
$$

However, we assume a "specific" functional form for $Y_i$ as a linear function of $\beta$ and $\tau$ as a constant. Generally, econometricians set the conditional expectation of potential outcome $Y_i(d)$, $d \in \{0,1\}$ as 

$$
E[Y_i(d) \mid X_i] = \mu_d(X_i).
$$ 

Hence, we can rewrite the ATE and ATT as follows:
$$
\tau_{\text{ATE}} = E\big[\mu_1(X_i) - \mu_0(X_i)\big], \qquad
\tau_{\text{ATT}} = E\big[Y_i - \mu_0(X_i) \mid D_i = 1\big].
$$

So, if we use a conventional regression model to estimate causal effects, using the sample analog, 

$$
\hat\tau_{\text{ATE}} = \frac{1}{n} \sum_{i=1}^{n} \big( \hat\mu_1(X_i) - \hat\mu_0(X_i) \big), \qquad
\hat\tau_{\text{ATT}} = \frac{1}{N^1} \sum_{i:\, D_i = 1} \big( Y_i - \hat\mu_0(X_i) \big).
$$

**Steps.**

1. Regress $Y$ on $X$ among the treated to get $\hat\mu_1$, and among the untreated to get $\hat\mu_0$.
2. Predict both potential outcomes for everyone.
3. Average them over the total units. 



For the ATT only $\hat\mu_0$ is needed: it predicts what each treated person would have earned without treatment.

## Lab 3 — Regression

### 3.1 In the simulated world

We use the simulation setup above: 

$$
Y_i(0) = 8 + 2 X_i + 1.5 X_i^2 + \varepsilon_i, \qquad
Y_i(1) = Y_i(0) + 2 + X_i.
$$
The true $\mu_0(x)$ is quadratic. We fit the outcome models twice: with $x$ and $x^2$ (correct), and with $x$ only (wrong).

In [ ]:
import statsmodels.api as sm

def fit_mu(design):
    """Predicted mu_1 and mu_0 for everyone, from OLS fitted separately in each group."""
    Z = sm.add_constant(design)
    mu1 = sm.OLS(y[d == 1], Z[d == 1]).fit().predict(Z)
    mu0 = sm.OLS(y[d == 0], Z[d == 0]).fit().predict(Z)
    return mu1, mu0

mu1_right, mu0_right = fit_mu(np.column_stack([x, x**2]))   # correct outcome model
mu1_wrong, mu0_wrong = fit_mu(x)                            # misspecified: no x^2

print("Regression estimator, ATE, correct model:", round((mu1_right - mu0_right).mean(), 3))
print("Regression estimator, ATE, wrong model:  ", round((mu1_wrong - mu0_wrong).mean(), 3))
print("True ATE:                                 ", round((y1 - y0).mean(), 3))

### 3.2 On the NSW–CPS data

For the ATT we only need $\hat\mu_0$: regress 1978 earnings on the covariates among the **CPS men**, then predict for the trainees.

In [ ]:
outcome_formula = "re78 ~ age + I(age**2) + educ + I(educ**2) + marr + nodegree + black + hisp + re74 + re75 + u74 + u75"
mu0_model = smf.ols(outcome_formula, data=c).fit()
mu0_t = mu0_model.predict(t)                       # predicted earnings without training, for each trainee

att_ra = (t["re78"] - mu0_t).mean()
print("Regression estimator, ATT:", round(att_ra, 1), "  (benchmark", round(benchmark, 1), ")")

**Question 4.** The outcome model is fitted on CPS men, most of whom look very different from the trainees (Lab 1). The prediction $\hat\mu_0(X_i)$ for a trainee is therefore an **extrapolation**. Why is this dangerous, and how does it relate to the overlap problem of IPW?

*Your answer to Question 4:*

## Doubly robust estimation: AIPW

Estimating causal parameters with a regression is wrong if the outcome model is wrong; IPW is wrong if the propensity score model is wrong. **Augmented IPW** combines them: start from the regression prediction and correct it with an IPW-weighted average of the prediction errors (residuals).

**ATE.**

$$
\hat\tau_{\text{AIPW}} = \frac{1}{n} \sum_{i=1}^{n} \left[ \hat\mu_1(X_i) - \hat\mu_0(X_i)
+ \frac{D_i \big(Y_i - \hat\mu_1(X_i)\big)}{\hat e(X_i)}
- \frac{(1 - D_i)\big(Y_i - \hat\mu_0(X_i)\big)}{1 - \hat e(X_i)} \right].
$$

**ATT.** With $\omega_i = \hat e_i / (1 - \hat e_i)$ as in IPW,

$$
\hat\tau_{\text{ATT}}^{\text{AIPW}} = \frac{1}{N^1} \sum_{i:\, D_i = 1} \big( Y_i - \hat\mu_0(X_i) \big)
- \frac{\sum_{i} (1 - D_i)\, \omega_i \big( Y_i - \hat\mu_0(X_i) \big)}{\sum_{i} (1 - D_i)\, \omega_i}.
$$

The first term is the regression estimator of causal effects; the second subtracts the weighted average residual of the untreated.

**Why "doubly robust".**

- If the **outcome model is right**, the residuals average to zero in every group, so the correction term vanishes on average, whatever the weights.
- If the **propensity score is right**, the weighted residuals estimate exactly the error that the regression model makes, so the correction removes it, whatever the outcome model.

So the estimator is consistent if **either** model is correct (Robins, Rotnitzky and Zhao, 1994; Bang and Robins, 2005).

## Lab 4 — AIPW

### 4.1 Doubly robust, in the simulated world

We combine a right or wrong propensity score model (logit with or without $x^2$) with a right or wrong outcome model (Lab 3.1).

In [ ]:
def fit_e(design):
    """Estimated propensity score from a logit of d on the columns of design."""
    Z = sm.add_constant(design)
    return sm.Logit(d, Z).fit(disp=0).predict(Z)

e_right = fit_e(np.column_stack([x, x**2]))   # correct propensity score model
e_wrong = fit_e(x)                            # misspecified: no x^2

def aipw_ate(e_hat, mu1, mu0):
    return np.mean(mu1 - mu0 + d * (y - mu1) / e_hat - (1 - d) * (y - mu0) / (1 - e_hat))

def ipw_ate(e_hat):
    w1, w0 = d / e_hat, (1 - d) / (1 - e_hat)
    return (w1 * y).sum() / w1.sum() - (w0 * y).sum() / w0.sum()

table = pd.DataFrame({
    "IPW": [ipw_ate(e_right), ipw_ate(e_wrong)],
    "AIPW, outcome model right": [aipw_ate(e_right, mu1_right, mu0_right), aipw_ate(e_wrong, mu1_right, mu0_right)],
    "AIPW, outcome model wrong": [aipw_ate(e_right, mu1_wrong, mu0_wrong), aipw_ate(e_wrong, mu1_wrong, mu0_wrong)],
}, index=["PS model right", "PS model wrong"]).round(3)
print("True ATE:", round((y1 - y0).mean(), 3))
table

**Question 5.** Which cells of the table are close to the true ATE? Relate each cell to the two bullet points above. What happens when both models are wrong?

*Your answer to Question 5:*

### 4.2 On the NSW–CPS data

The ATT version, with the flexible propensity score of Lab 1 and the outcome model of Lab 3.2.

In [ ]:
mu0_c = mu0_model.predict(c)                       # predicted earnings of the CPS men (their fitted values)

att_aipw = (t["re78"] - mu0_t).mean() - (omega * (c["re78"] - mu0_c)).sum() / omega.sum()
print("AIPW, ATT:", round(att_aipw, 1), "  (benchmark", round(benchmark, 1), ")")

## Lab 5 — Propensity score matching (optional)

Nearest-neighbor matching from Week 3, but on one number: for each trainee $i$, find the CPS man $j(i)$ with the closest estimated score, and average the differences:

$$
\hat\tau_{\text{ATT}} = \frac{1}{N^1} \sum_{i:\, D_i = 1} \big( Y_i - Y_{j(i)} \big).
$$

In [ ]:
e1 = t["e"].to_numpy()
e0 = c["e"].to_numpy()
j_ps = np.abs(e1[:, None] - e0[None, :]).argmin(axis=1)     # closest CPS man in the score, for each trainee

att_psm = (t["re78"].to_numpy() - c["re78"].to_numpy()[j_ps]).mean()
print("PS matching, ATT:", round(att_psm, 1), "   distinct CPS men used:", len(np.unique(j_ps)))

Did matching on the score balance the covariates? We use the standardized difference from Weeks 2 and 3, keeping the denominator from before matching.

In [ ]:
sd_pooled = np.sqrt((t[covariates].var() + c[covariates].var()) / 2)
matched = c.iloc[j_ps]

pd.DataFrame({
    "before": (t[covariates].mean() - c[covariates].mean()) / sd_pooled,
    "after PS matching": (t[covariates].mean() - matched[covariates].mean().to_numpy()) / sd_pooled,
}).round(2)

**Question 6.** Compare the balance with the Mahalanobis matching of Week 3 (all standardized differences within $\pm 0.02$). Which is better balanced, and why might matching on one estimated number balance the covariates less well than matching on the covariates themselves? (See King and Nielsen, 2019.)

*Your answer to Question 6:*

**Note.** Standard errors for matching on an **estimated** score must account for the estimation of the score (Abadie and Imbens, 2016). We do not compute them here.

## Lab 6 — Let a package do it

We wrote every estimator ourselves. `statsmodels` also has them in one class, `TreatmentEffect` (in `statsmodels.treatment.treatment_effects`), with the methods `ipw`, `ra` and `aipw` (`ra` stands for regression adjustment, another name for the regression estimator of Lab 3). The option `effect_group=1` gives the effect on the treated (ATT).

Ask your AI assistant to write the code. For example:

> Using `TreatmentEffect` from `statsmodels.treatment.treatment_effects`, estimate treatment effects in two data sets that are already in my Python session.
> (1) Arrays `y`, `d`, `x`: estimate the ATE by IPW, the regression estimator and AIPW, with the propensity score model `d ~ x + I(x**2)` (logit) and the outcome model `y ~ x + I(x**2)` (OLS).
> (2) A data frame `both` with outcome `re78` and treatment `d`: estimate the ATT by IPW and the regression estimator. The fitted logit for the propensity score is called `logit`, and the outcome model formula is the string `outcome_formula`.
> Store the five point estimates in a list called `package`, in this order: the three ATEs from (1), then the two ATTs from (2).

Paste the code into the `# TODO` cell below and run it. Read it before you trust it: which model is used for the propensity score and which for the outcome?

In [ ]:
# TODO: replace `package = None` with the code from your AI assistant.
# The code must create the list `package` with the five estimates, in the order of the prompt.
package = None

In [ ]:
# Check (do not edit)
if package is None:
    print("not done yet")
else:
    by_hand = [ipw_ate(e_right), (mu1_right - mu0_right).mean(), aipw_ate(e_right, mu1_right, mu0_right),
               att_ipw_nsw, att_ra]
    print(pd.DataFrame({"by hand": by_hand, "package": list(package)},
                       index=["Simulation: IPW, ATE", "Simulation: RA, ATE", "Simulation: AIPW, ATE",
                              "NSW-CPS: IPW, ATT", "NSW-CPS: RA, ATT"]).round(3))
    print("all five match" if np.allclose(by_hand, list(package), rtol=1e-3) else "some numbers differ: see Question 7")

**Question 7.** Do the package numbers agree with yours? `aipw` in `TreatmentEffect` has no `effect_group` option: which of our estimates can you not reproduce with it? If a package gives a different number from your own code, how would you find out which one is right?

*Your answer to Question 7:*

## Summary

In [ ]:
naive = t["re78"].mean() - c["re78"].mean()
pd.Series({
    "Experimental benchmark": benchmark,
    "Naive NSW - CPS": naive,
    "IPW": att_ipw_nsw,
    "Regression estimator": att_ra,
    "AIPW": att_aipw,
}, name="ATT").round(1)

**Question 8.** Which estimates are close to the benchmark? None of them comes with a standard error here. What would you need before concluding that one method "works" on these data?

*Your answer to Question 8:*